# Run training sufficient for inference
This notebook does not introduce new functionality. It uses the extracted, refactored python modules to run extensive training that will be sufficient for inference.

Alternatively, the same training can be accomplished by using the CLI scripts (`nanollm-train`, `nanollm-resume`), which use the same underlying python modules.

In [ ]:
# initialize structure to capture loss history across phases
all_losses = []

## Run initial training phase

In [ ]:
from src.config import ModelConfig, TokenizerConfig, TrainingConfig
from src.paths import DEFAULT_DATA_FILE
from src.training.checkpoint import default_checkpoint_path
from src.training.runner import Runner

# Define configs
model_config = ModelConfig()
tokenizer_config = TokenizerConfig()
TRAINING_EPOCHS = 25 # will be used for all phases
training_config = TrainingConfig(epochs=TRAINING_EPOCHS)

# Phase 1: 
#   Runner builds a fresh model from provided configurations and persists the initial checkpoint
#   NB: By omitting the optional parameter (checkpoint_source), the runner will construct a fresh model from the provided configs instead of loading pre-trained weights.
runner = Runner(
    model_config=model_config,
    tokenizer_config=tokenizer_config,
    data_source=DEFAULT_DATA_FILE,
    training_config=training_config,
    checkpoint_destination=default_checkpoint_path(),
)

# Runner returns metrics history logged by epoch
history = runner.run()
print(f"Phase 1: {history}")

# Append to global structure for visualization in this notebook of training over all epochs
all_losses.extend(history.train_loss)

## Resume training in multiple phases

In [ ]:
from src.training.checkpoint import get_latest_checkpoint

# Subsequent Phases: each loop loads the most recently persisted checkpoint, orchestrates training, and persists a new checkpoint
PHASES = 3
for phase in range(PHASES):
    most_recent_checkpoint = get_latest_checkpoint()

    #   Runner builds a pre-trained model using weights from the checkpoint_source path. It also loads the metadata from that checkpoint, 
    #   which is required for things like calculating the cumulative number of epochs trained. 
    runner = Runner(
        data_source=DEFAULT_DATA_FILE,
        training_config=training_config,
        checkpoint_destination=default_checkpoint_path(), # execute on each run to generate new timestamped directory name
        checkpoint_source=most_recent_checkpoint,         # if this parameter is defined, we don't have to pass in model_config and tokenizer_config
    )

    history = runner.run()
    print(f"Phase {phase + 2}: {history}")

    # append for visualization over all epochs
    all_losses.extend(history.train_loss)

## Visualize Training

In [ ]:
import pprint
from src.training.checkpoint import load_metadata

# load cumulative training metrics
most_recent_checkpoint = get_latest_checkpoint()
pprint.pp(load_metadata(most_recent_checkpoint))                                                                                     

In [ ]:
import matplotlib.pyplot as plt

plt.plot(all_losses)
plt.title('Training Loss (All Epochs)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()